# DUS Simulatie Fryslân

## RUG x DataFryslân Field Project 2026

Deze simulatie heeft als doel om met beschikbare data van het transportnetwerk van de provincie Fryslân, het woon-werkverkeer binnen de provincie en van/naar omliggende provincies te simuleren. Het maakt gebruik van de volgende databronnen:
- CBS Woon-Werkverkeer in Friesland (2024)
- CBS Kerncijfers Wijken en Buurten (2024)
- CBS Gebiedsindelingen (2024)
- General Transit Feed Specification (GTFS)nl OVapi
- Open Street Map (OSM)

Meer over de databronnen is te vinden in de README.md

**Belangrijk**: Pas in de *Libaries & Setup* sectie de paden aan naar de GTFS & CBS gebiedsindelingen data aan, evenals het pad naar de validatie dataset (meer daarover in stap 7)

## Inhoudsopgave:

### 1. Libraries & Setup
### 2. Data Importeren
### 3. Gravity Model & Logit Modal Split
### 4. OV Simulatie
### 5. Auto Simulatie
### 6. Validatie
### 7. Interactieve Kaart
### 8. Resultaten
### 9. Sterke en Zwakke Punten

### 1. Libraries & Setup

In [ ]:
# Libraries & Setup
import pandas as pd
import numpy as np
import geopandas as gpd
import simpy
import random
from datetime import datetime, timedelta
import fiona
import cbsodata
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
from tqdm import tqdm
from shapely.geometry import Point
import folium
from folium import plugins
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import os
import heapq
from collections import Counter
import gc
import csv
import bisect

In [ ]:
# Paden instellen -> VERANDER NAAR EIGEN LOCATIES
path_to_gpkg = '/Users/jurre/Documents/BSc Data Science & Society/Year 2/Field Project/cbsgebiedsindelingen2016_heden/cbsgebiedsindelingen2024.gpkg' 
path_to_gtfs = '/Users/jurre/Documents/BSc Data Science & Society/Year 2/Field Project/gtfs-nl'
path_to_val = '/Users/jurre/Documents/BSc Data Science & Society/Year 2/Field Project/avondspits.csv'

In [ ]:
# Deze cel bevat alle belangrijke instellingen en parameters van de simulatie
# Pas deze waarden aan om scenario's door te rekenen

# A. Demografie & Tijdsinstellingen
SIM_DATE = "20260527"          # Dinsdag 27 mei 2026
GROEI_FACTOR = 1.00572         # Bevolkingsgroei factor (CBS dec 2024 -> apr 2026 = +0,572%; Bron: CBS Bevolkingsontwikkeling; maand en jaar)
SIMULATIE_SCHAAL_FACTOR = 0.05 # Simulatie grootte beperken tot 0.5%
AANDEEL_SPITSUUR_AUTO = 0.10   # Geschat percentage van het dagelijks autoverkeer dat in 1 spitsuur valt (validatie)

# B. Kosten
# Bron: Nibud/CBS gemiddeldes
KOSTEN_AUTO_PER_KM = 0.21      # Euro per kilometer
KOSTEN_OV_PER_KM = 0.20        # Euro per kilometer

# C. Gravity Model & CBS Data
BETA = 1.5                                  # Vervalparameter β Gravity Model (weerstand tegen afstand)
KOLOM_PENDELAARS = 'BanenVanWerknemers_1'   # CBS Dataset kolomnaam voor pendelvolume
KOLOM_AFSTAND = 'WoonWerkafstand_2'         # CBS Dataset kolomnaam voor afstand
BUURT_LAAG_NAAM = 'buurt_labelpoint' 

# D. Logit Modal Split
# Alfa parameters: Base utility/voorkeur onafhankelijk van tijd/kosten
ALPHA_AUTO = 2.0
ALPHA_OV = 0.0
ALPHA_FIETS = 1.5
ALPHA_LOPEN = 0.5

# Bèta parameters: Gevoeligheid voor Tijd en Geld (Negatief = Straf)
BETA_TIJD = -0.05              # Elke minuut reistijd doet pijn
BETA_KOSTEN = -0.15            # Elke euro doet meer pijn

# In de Simulatie wordt geen rekening gehouden met de grote hoeveelheden studenten die tussen Friesland en Groningen reizen
# Deze kunnen handcoded worden bijgeschreven
VOLUME_GR_LWD = 0              # Volume OV reizigers op het OV traject Groningen - Leeuwarden wordt handmatig toegevoegd
VOLUME_LWD_GR = 0              # Volume OV reizigers op het OV traject Leeuwarden - Groningen wordt handmatig toegevoegd

START_TIME = 21600             # Starttijd simulatie: 06:00 (21600 sec)

In [ ]:
# Seed voor reproduceerbaarheid
random.seed(42)
np.random.seed(42)

### 2. Data Importeren

In deze stap wordt de benodigde data ingeladen.
De benodigde data bevat: Populatiecijfers, pendelcijfers, dienstregelingen, wegennetwerken, etc.

In [ ]:
#  Data Importeren (CBS & Geo)

municipalities = gpd.read_file(path_to_gpkg, layer='gemeente_gegeneraliseerd')
municipalities['geometry'] = municipalities.geometry.make_valid()

# Hardcoded lijst gemeenten Fryslân
gemeenten = [
    "Achtkarspelen", "Ameland", "Dantumadiel", "De Fryske Marren", 
    "Harlingen", "Heerenveen", "Leeuwarden", "Noardeast-Fryslân", 
    "Ooststellingwerf", "Opsterland", "Schiermonnikoog", "Smallingerland", 
    "Súdwest-Fryslân", "Terschelling", "Tytsjerksteradiel", "Vlieland", 
    "Waadhoeke", "Weststellingwerf"
]

Fryslân = municipalities[municipalities['statnaam'].isin(gemeenten)].copy()

# CBS Tabel 85481NED Woon-Werkverkeer in Fryslân (2024)
table_id = '85481NED'
regio_meta = pd.DataFrame(cbsodata.get_meta(table_id, 'WoonregioS'))
extra_gemeenten = ['Groningen', 'Zwolle', 'Alkmaar', 'Noordoostpolder', 'Assen', 'Noordenveld', 'Steenwijkerland', 'Westerkwartier']
alle_toegestane_gemeenten = gemeenten + extra_gemeenten
cbs_zoek_gemeenten = alle_toegestane_gemeenten + ['Groningen (gemeente)']
friesland_keys_exact = regio_meta[regio_meta['Title'].str.strip().isin(cbs_zoek_gemeenten)]['Key'].tolist()

woon_filter = " or ".join([f"WoonregioS eq '{k}'" for k in friesland_keys_exact])
werk_filter = " or ".join([f"WerkregioS eq '{k}'" for k in friesland_keys_exact])
periode_filter = "Perioden eq '2024MM12'"
full_filter = f"({woon_filter}) and ({werk_filter}) and {periode_filter}"

pendel_matrix = pd.DataFrame(cbsodata.get_data(table_id, filters=full_filter, select=['WoonregioS', 'WerkregioS', 'Perioden', 'BanenVanWerknemers_1','WoonWerkafstand_2']))

# CBS Tabel 85984NED: Kerncijfers Wijken en Buurten (2024)
df_alles = pd.DataFrame(cbsodata.get_data('85984NED'))
col_regio_type = [c for c in df_alles.columns if 'SoortRegio' in c or 'soort' in c.lower()][0]
col_code       = [c for c in df_alles.columns if 'Codering' in c or 'code' in c.lower()][0]
col_gemeente   = [c for c in df_alles.columns if 'Gemeentenaam' in c or 'gemeente' in c.lower()][0]
col_inwoners   = [c for c in df_alles.columns if 'AantalInwoners' in c or 'inwoners' in c.lower()][0]

df_buurten = df_alles[df_alles[col_regio_type].str.strip() == 'Buurt'].copy()
df_buurten['Gemeente_Lower'] = df_buurten[col_gemeente].str.lower().str.strip()
friese_gemeenten_clean = [g.lower().strip() for g in alle_toegestane_gemeenten]

df_wijkenenbuurten = df_buurten[df_buurten['Gemeente_Lower'].isin(friese_gemeenten_clean)].copy()
df_wijkenenbuurten = df_wijkenenbuurten[[col_code, 'WijkenEnBuurten', col_gemeente, col_inwoners]].rename(
    columns={col_code: 'buurt_code', 'WijkenEnBuurten': 'buurt_naam', col_gemeente: 'gemeente_naam', col_inwoners: 'aantal_inwoners'})
df_wijkenenbuurten['aantal_inwoners'] = df_wijkenenbuurten['aantal_inwoners'].fillna(0).astype(int)

# Geolocatie Buurten (GPKG)
kaarten = gpd.read_file(path_to_gpkg, layer=BUURT_LAAG_NAAM)
gpkg_code_kolom = 'statcode'
kaarten[gpkg_code_kolom] = kaarten[gpkg_code_kolom].astype(str).str.strip()
df_wijkenenbuurten['buurt_code'] = df_wijkenenbuurten['buurt_code'].astype(str).str.strip()
buurten_fryslan_gdf = kaarten.merge(df_wijkenenbuurten, left_on=gpkg_code_kolom, right_on='buurt_code', how='inner')

print('CBS Data succesvol ingeladen.')

In de volgende stap wordt de GTFS data voorbereid voorbereid voor simulatie. Dit resulteert in het dataframe 'friese_ov_stations'

In [ ]:
# GTFS Data & OV Haltes Filteren

routes = pd.read_csv(os.path.join(path_to_gtfs, 'routes.txt'), dtype=str)
trips = pd.read_csv(os.path.join(path_to_gtfs, 'trips.txt'), dtype=str)
stop_times = pd.read_csv(os.path.join(path_to_gtfs, 'stop_times.txt'), dtype=str)
stops = pd.read_csv(os.path.join(path_to_gtfs, 'stops.txt'), dtype=str)
calendar_dates = pd.read_csv(os.path.join(path_to_gtfs, 'calendar_dates.txt'))

ov_routes = routes[routes['route_type'].isin([2, 3, 700, 701, 702, 703, 704, '2', '3', '700'])]['route_id'].unique()
ov_trips = trips[trips['route_id'].isin(ov_routes)]['trip_id'].unique()
ov_stop_times = stop_times[stop_times['trip_id'].isin(ov_trips)].dropna(subset=['arrival_time'])
ov_stations = stops[stops['stop_id'].isin(ov_stop_times['stop_id'].unique())]

ov_stations_gdf = gpd.GeoDataFrame(ov_stations, geometry=gpd.points_from_xy(ov_stations.stop_lon.astype(float), ov_stations.stop_lat.astype(float)), crs="EPSG:4326")
Fryslân = Fryslân.to_crs(crs="EPSG:4326")

friese_ov_stations_puur = gpd.sjoin(ov_stations_gdf, Fryslân, predicate='within')
friese_station_ids_spatial = set(friese_ov_stations_puur['stop_id'].tolist())

extra_bus_lijnen = ['315', '324', '350', '39', '133', '139', '101', '84', '104', '18', '217']
extra_routes = routes[(routes['route_short_name'].isin(extra_bus_lijnen)) | (routes['agency_id'].isin(['WPD', 'DOEKSEN'])) | ((routes['route_type'].isin([2, '2'])) & (routes['route_long_name'].str.contains('Zwolle|Groningen|Leeuwarden', na=False)))]['route_id'].astype(str)
extra_trips = trips[trips['route_id'].astype(str).isin(extra_routes)]['trip_id'].astype(str)
extra_stops = stop_times[stop_times['trip_id'].astype(str).isin(extra_trips)]['stop_id'].unique()

alle_ov_stations = ov_stations_gdf[ov_stations_gdf['stop_id'].isin(friese_station_ids_spatial.union(extra_stops))]
friese_ov_stations = alle_ov_stations.copy()

train_routes = routes[routes['route_type'].isin([2, '2'])]['route_id']
train_trips = trips[trips['route_id'].isin(train_routes)]['trip_id']
train_stop_ids = stop_times[stop_times['trip_id'].isin(train_trips)]['stop_id'].unique()

friese_train_stations = friese_ov_stations[friese_ov_stations['stop_id'].isin(train_stop_ids)].copy()
friese_bus_stations = friese_ov_stations[~friese_ov_stations['stop_id'].isin(train_stop_ids)].copy()

if not friese_train_stations.empty:
    friese_train_stations['geometry'] = friese_train_stations.geometry.make_valid()
    bus_gdf_metric = friese_bus_stations.to_crs("EPSG:28992")
    train_gdf_metric = friese_train_stations.to_crs("EPSG:28992")
    train_gdf_metric['geometry'] = train_gdf_metric.geometry.buffer(250)
    
    bus_gdf_metric = bus_gdf_metric.drop(columns=['index_right', 'index_left'], errors='ignore')
    train_gdf_metric = train_gdf_metric.drop(columns=['index_right', 'index_left'], errors='ignore')
    
    joined = gpd.sjoin(bus_gdf_metric, train_gdf_metric[['stop_name', 'geometry']], how='left', predicate='intersects')
    joined['stop_name'] = joined['stop_name_right'].fillna(joined['stop_name_left'])
    updated_bus = joined.drop(columns=['index_right', 'stop_name_left', 'stop_name_right']).to_crs("EPSG:4326")
    friese_ov_stations = pd.concat([friese_train_stations, updated_bus]).drop_duplicates(subset=['stop_id'])
friese_ov_stations.head(5)

In de volgende stap wordt het Open Street Map wegennetwerk ingeladen. In de Github Repository staan twee .graphml bestanden met een cache van het ingeladen netwerk voor deze simulatie. Dit zorgt ervoor dat de API niet nogmaals dit uitgebreide netwerk van Noord-Nederland hoeft op te halen.

In [ ]:
# OSMnx Netwerk Ophalen

ox.settings.log_console = False

# Cache bestanden
cache_file_drive = "netwerk_auto_friesland_provincies.graphml"
cache_file_bike = "netwerk_fiets_friesland.graphml"

# Autonetwerk (G_drive)
if os.path.exists(cache_file_drive):
    # Cache inladen
    G_drive = ox.load_graphml(cache_file_drive)
else:    
    # Friesland hoofdwegennetwerk opbouwen via API
    G_fryslan_raw = ox.graph_from_place({"state": "Friesland", "country": "Netherlands"}, network_type="drive")
    largest_cc = max(nx.strongly_connected_components(G_fryslan_raw), key=len)
    G_fryslan = G_fryslan_raw.subgraph(largest_cc).copy()
    
    custom_filter = '["highway"~"motorway|motorway_link|trunk|trunk_link|primary|primary_link|secondary|secondary_link"]'
    G_provincies = ox.graph_from_place([
        {'state': 'Friesland', 'country': 'Netherlands'},
        {'state': 'Groningen', 'country': 'Netherlands'},
        {'state': 'Drenthe', 'country': 'Netherlands'},
        {'state': 'Overijssel', 'country': 'Netherlands'},
        {'state': 'Flevoland', 'country': 'Netherlands'},
        {'state': 'Noord-Holland', 'country': 'Netherlands'}
    ], custom_filter=custom_filter)
    
    # Auto netwerk samenvoegen
    G_drive = nx.compose(G_fryslan, G_provincies)
    G_drive = ox.add_edge_speeds(G_drive)
    G_drive = ox.add_edge_travel_times(G_drive)
    
    # Cache opslaan voor de volgende keer
    ox.save_graphml(G_drive, cache_file_drive)
    print(f"Autonetwerk gedownload en opgeslagen als '{cache_file_drive}'.")

# Fietsnetwerk (G_bike)
if os.path.exists(cache_file_bike):
    # Cache inladen
    G_bike = ox.load_graphml(cache_file_bike)
else:
    # API bebruiken om het netwerk in te laden
    G_bike = ox.graph_from_place("Friesland, Netherlands", network_type="bike")
    G_bike = ox.add_edge_speeds(G_bike, fallback=15)
    G_bike = ox.add_edge_travel_times(G_bike)
    
    # Cache opslaan voor de volgende keer
    ox.save_graphml(G_bike, cache_file_bike)
    print(f"Fietsnetwerk gedownload en opgeslagen als '{cache_file_bike}'.")

### 3. Gravity Model & Logit Modal Split

In deze stap worden het Gravity Model (Zwaartekracht Model) en de Modal Split toegepast volgens de formules beschreven in het onderzoek. De waarden van deze formules kunnen worden aangepast met de parameters in hoofdstuk 1.

In [ ]:
# Gravity Model & Modal Split
df_pendel = pendel_matrix.copy()
df_pendel['WoonregioS'] = df_pendel['WoonregioS'].str.replace(' (gemeente)', '', regex=False).str.strip()
df_pendel['WerkregioS'] = df_pendel['WerkregioS'].str.replace(' (gemeente)', '', regex=False).str.strip()
df_pendel = df_pendel[((df_pendel['WoonregioS'].isin(gemeenten)) & (df_pendel['WerkregioS'].isin(alle_toegestane_gemeenten))) | ((df_pendel['WoonregioS'].isin(alle_toegestane_gemeenten)) & (df_pendel['WerkregioS'].isin(gemeenten)))]
df_pendel[KOLOM_PENDELAARS] = df_pendel[KOLOM_PENDELAARS] * 1000    # CBS dataset is x1000

massa_oorsprong = df_pendel.groupby('WoonregioS')[KOLOM_PENDELAARS].sum().reset_index().rename(columns={'WoonregioS': 'Gemeente', KOLOM_PENDELAARS: 'Y_i'})
massa_bestemming = df_pendel.groupby('WerkregioS')[KOLOM_PENDELAARS].sum().reset_index().rename(columns={'WerkregioS': 'Gemeente', KOLOM_PENDELAARS: 'Y_j'})

gravity_model = df_pendel[['WoonregioS', 'WerkregioS', KOLOM_AFSTAND, KOLOM_PENDELAARS]].copy().rename(columns={KOLOM_AFSTAND: 'Afstand_km', KOLOM_PENDELAARS: 'Werkelijke_Pendel'})
gravity_model = gravity_model.merge(massa_oorsprong, left_on='WoonregioS', right_on='Gemeente', how='inner').drop(columns=['Gemeente'])
gravity_model = gravity_model.merge(massa_bestemming, left_on='WerkregioS', right_on='Gemeente', how='inner').drop(columns=['Gemeente'])

gravity_model['Afstand_km'] = pd.to_numeric(gravity_model['Afstand_km'], errors='coerce')
gravity_model = gravity_model.dropna(subset=['Afstand_km'])
gravity_model = gravity_model[gravity_model['Afstand_km'] > 0].copy()

gravity_model['Theoretische_Interactie'] = ((gravity_model['Y_i'] * gravity_model['Y_j']) / (gravity_model['Afstand_km'] ** BETA))
schaalfactor = gravity_model['Werkelijke_Pendel'].sum() / gravity_model['Theoretische_Interactie'].sum()

gravity_model['DUS_Reizigers_Totaal'] = gravity_model['Werkelijke_Pendel']
fallback = (gravity_model['Theoretische_Interactie'] * schaalfactor).round().astype(int)
gravity_model.loc[(gravity_model['DUS_Reizigers_Totaal'].isnull()) | (gravity_model['DUS_Reizigers_Totaal'] == 0), 'DUS_Reizigers_Totaal'] = fallback

# Reistijden ophalen uit de G_drive/G_bike netwerken
fryslan_copy = Fryslân.to_crs(epsg=28992).copy()
fryslan_copy['centroid'] = fryslan_copy.geometry.centroid
centroids_wgs84 = fryslan_copy['centroid'].to_crs(epsg=4326)

gemeente_nodes_car, gemeente_nodes_bike = {}, {}
for statnaam, point in zip(fryslan_copy['statnaam'], centroids_wgs84):
    gemeente_nodes_car[statnaam] = ox.distance.nearest_nodes(G_drive, X=point.x, Y=point.y)
    gemeente_nodes_bike[statnaam] = ox.distance.nearest_nodes(G_bike, X=point.x, Y=point.y)

externe_gemeenten = set(gravity_model['WoonregioS'].unique()).union(set(gravity_model['WerkregioS'].unique())) - set(fryslan_copy['statnaam'])
for gem in externe_gemeenten:
    try:
        lat, lon = ox.geocode(gem.strip() + ", Netherlands")
        gemeente_nodes_car[gem] = ox.distance.nearest_nodes(G_drive, X=lon, Y=lat)
        gemeente_nodes_bike[gem] = ox.distance.nearest_nodes(G_bike, X=lon, Y=lat)
    except: pass

reistijden_auto, reistijden_fiets = [], []

for row in tqdm(list(gravity_model.itertuples()), desc="OD Reistijden Berekenen"):
    woon, werk = row.WoonregioS.strip(), row.WerkregioS.strip()
    tt_car, tt_bike = np.nan, np.nan
    if woon in gemeente_nodes_car and werk in gemeente_nodes_car:
        try: tt_car = nx.shortest_path_length(G_drive, gemeente_nodes_car[woon], gemeente_nodes_car[werk], weight='travel_time') / 60.0
        except nx.NetworkXNoPath: pass
    if woon in gemeente_nodes_bike and werk in gemeente_nodes_bike:
        try: tt_bike = nx.shortest_path_length(G_bike, gemeente_nodes_bike[woon], gemeente_nodes_bike[werk], weight='travel_time') / 60.0
        except nx.NetworkXNoPath: pass
    reistijden_auto.append(tt_car)
    reistijden_fiets.append(tt_bike)

gravity_model['Reistijd_Auto_min'] = reistijden_auto
gravity_model['Reistijd_Fiets_min'] = reistijden_fiets

# Logit Modal Split
modal_split = gravity_model.copy()
modal_split['Kosten_Auto'] = modal_split['Afstand_km'] * KOSTEN_AUTO_PER_KM
modal_split['Tijd_Auto'] = modal_split['Reistijd_Auto_min'].fillna((modal_split['Afstand_km'] / 80.0) * 60)
modal_split['Kosten_Fiets'] = 0.0
modal_split['Tijd_Fiets'] = modal_split['Reistijd_Fiets_min'].fillna((modal_split['Afstand_km'] / 15.0) * 60)
modal_split.loc[modal_split['Afstand_km'] > 15, 'Tijd_Fiets'] += 999 
modal_split['Kosten_Lopen'] = 0.0
modal_split['Tijd_Lopen'] = (modal_split['Afstand_km'] / 5.0) * 60 
modal_split.loc[modal_split['Afstand_km'] > 4, 'Tijd_Lopen'] += 9999 
modal_split['Kosten_OV'] = modal_split['Afstand_km'] * KOSTEN_OV_PER_KM
modal_split['Tijd_OV'] = (modal_split['Afstand_km'] / 70.0) * 60 + 10
modal_split.loc[(modal_split['WoonregioS'].str.contains('Leeuwarden')) & (modal_split['WerkregioS'].str.contains('Groningen')), 'Tijd_OV'] = 35
modal_split.loc[(modal_split['WoonregioS'].str.contains('Groningen')) & (modal_split['WerkregioS'].str.contains('Leeuwarden')), 'Tijd_OV'] = 35 

modal_split['V_Auto'] = np.clip(ALPHA_AUTO + (BETA_TIJD * modal_split['Tijd_Auto']) + (BETA_KOSTEN * modal_split['Kosten_Auto']), -100, None)
modal_split['V_OV'] = np.clip(ALPHA_OV + (BETA_TIJD * modal_split['Tijd_OV']) + (BETA_KOSTEN * modal_split['Kosten_OV']), -100, None)
modal_split['V_Fiets'] = np.clip(ALPHA_FIETS + (BETA_TIJD * modal_split['Tijd_Fiets']) + (BETA_KOSTEN * modal_split['Kosten_Fiets']), -100, None)
modal_split['V_Lopen'] = np.clip(ALPHA_LOPEN + (BETA_TIJD * modal_split['Tijd_Lopen']) + (BETA_KOSTEN * modal_split['Kosten_Lopen']), -100, None)

som_exp = np.exp(modal_split['V_Auto']) + np.exp(modal_split['V_OV']) + np.exp(modal_split['V_Fiets']) + np.exp(modal_split['V_Lopen'])
modal_split['P_Auto'] = np.exp(modal_split['V_Auto']) / som_exp
modal_split['P_OV'] = np.exp(modal_split['V_OV']) / som_exp
modal_split['P_Fiets'] = np.exp(modal_split['V_Fiets']) / som_exp
modal_split['P_Lopen'] = np.exp(modal_split['V_Lopen']) / som_exp

modal_split['Forenzen_Auto'] = (modal_split['DUS_Reizigers_Totaal'] * modal_split['P_Auto']).fillna(0).round().astype(int)
modal_split['Forenzen_OV'] = (modal_split['DUS_Reizigers_Totaal'] * modal_split['P_OV']).fillna(0).round().astype(int)
modal_split['Forenzen_Fiets'] = (modal_split['DUS_Reizigers_Totaal'] * modal_split['P_Fiets']).fillna(0).round().astype(int)
modal_split['Forenzen_Lopen'] = (modal_split['DUS_Reizigers_Totaal'] * modal_split['P_Lopen']).fillna(0).round().astype(int)

# Hardcode Groningen-Leeuwarden OV volume direct to Forenzen_OV
modal_split.loc[(modal_split['WoonregioS'].str.contains('Leeuwarden')) & (modal_split['WerkregioS'].str.contains('Groningen')), 'Forenzen_OV'] += VOLUME_LWD_GR
modal_split.loc[(modal_split['WoonregioS'].str.contains('Groningen')) & (modal_split['WerkregioS'].str.contains('Leeuwarden')), 'Forenzen_OV'] += VOLUME_GR_LWD

top_ov = modal_split[modal_split['WoonregioS'] != modal_split['WerkregioS']].sort_values(by='Forenzen_OV', ascending=False)

In [ ]:
# Data opschalen naar 2026
simulatie_vraag = top_ov.copy()

# De kolommen die opgeschaald moeten worden
cols_to_scale = ['DUS_Reizigers_Totaal', 'Forenzen_Auto', 'Forenzen_OV', 'Forenzen_Fiets', 'Forenzen_Lopen']

for col in cols_to_scale:
    # Vermenigvuldigen en afronden naar hele personen
    simulatie_vraag[col] = (simulatie_vraag[col] * GROEI_FACTOR).round().astype(int)

# Bereken het verschil om te laten zien
oude_ov = top_ov['Forenzen_OV'].sum()
nieuwe_ov = simulatie_vraag['Forenzen_OV'].sum()
extra_passagiers = nieuwe_ov - oude_ov

print(f"{extra_passagiers} extra dagelijkse OV-passagiers gegenereerd.")
print("\nDe (externe) top-routes voor OV:")
print(simulatie_vraag[['WoonregioS', 'WerkregioS', 'Forenzen_OV']].head(5).to_string(index=False))

### 4. OV Simulatie

In dit hoofdstuk wordt de OV Simulatie voorbereid en gedraaid.

In de volgende code cel worden stations aan buurten gekoppeld. Daarna krijgen de stations per gekoppelde buurt gewichten, op basis waarvan virtuele passagiers instappen.

In [ ]:
# Ruimtelijke Koppeling (Buurtniveau)

# Coördinatensystemen (EPSG:28992 voor afstanden in meters)
stations_proj = friese_ov_stations.to_crs(epsg=28992)
buurten_proj = buurten_fryslan_gdf.to_crs(epsg=28992)

buurten_proj['gemeente_naam'] = buurten_proj['gemeente_naam'].str.strip()

# Buurten aan stations matchen
buurten_proj['Nearest_Station_ID'] = ""
buurten_proj['Nearest_Station_Name'] = ""

for idx, buurt in buurten_proj.iterrows():
    distances = stations_proj.geometry.distance(buurt.geometry)
    nearest_idx = distances.idxmin()
    buurten_proj.at[idx, 'Nearest_Station_ID'] = stations_proj.loc[nearest_idx, 'stop_id']
    buurten_proj.at[idx, 'Nearest_Station_Name'] = stations_proj.loc[nearest_idx, 'stop_name']

# Station-gewichten berekenen op basis van populatie
print("Stations-gewichten berekenen per gemeente op basis van inwoneraantal...")
station_weights = buurten_proj.groupby(['gemeente_naam', 'Nearest_Station_Name'])['aantal_inwoners'].sum().reset_index()

gemeente_totals = station_weights.groupby('gemeente_naam')['aantal_inwoners'].sum().reset_index()
gemeente_totals = gemeente_totals.rename(columns={'aantal_inwoners': 'totaal_gemeente_inwoners'})

station_weights = station_weights.merge(gemeente_totals, on='gemeente_naam')
station_weights['totaal_gemeente_inwoners'] = station_weights['totaal_gemeente_inwoners'].replace(0, 1) 
station_weights['Station_Weight'] = station_weights['aantal_inwoners'] / station_weights['totaal_gemeente_inwoners']

print(station_weights[station_weights['gemeente_naam'] == 'Tytsjerksteradiel'][['gemeente_naam', 'Nearest_Station_Name', 'Station_Weight']].to_string(index=False))

In de volgende cel worden de virtuele (OV-)Reizigers gegenereerd op basis van de station/halte weights. Ook wordt het aantal passagiers vermenigvuldigd met een schaal factor, die er voor zorgt dat slechts een paar procent van de gehele groep passagiers wordt gesimuleerd over het netwerk (ivm rekenkracht, geheugen, etc.). Ook bepaald deze cel op welk tijdstip de reizigers worden gegeneerd, omdat ze natuurlijk niet geleidelijk over de dag over het netwerk reizen. Deze parameters zijn bovenin te vinden en aan te passen.

In [ ]:
# Reizigers Generatie en Tijdsprofielen (Heenreis + Terugreis)

# Parameters tijdsprofielen
P_HEEN_OCHTEND = 0.70   # 70% van de heenreizigers reist in de ochtendspits
P_HEEN_DAL = 0.25
P_HEEN_AVOND = 0.05

P_TERUG_OCHTEND = 0.05
P_TERUG_DAL = 0.25
P_TERUG_AVOND = 0.70

passagiers_lijst = []
simulatie_vraag['WoonregioS'] = simulatie_vraag['WoonregioS'].str.strip()
simulatie_vraag['WerkregioS'] = simulatie_vraag['WerkregioS'].str.strip()

# Passagiers genereren op basis van de populatie-gewogen koppeling
for idx, route in simulatie_vraag.iterrows():
    origin = route['WoonregioS']
    dest = route['WerkregioS']
    totaal_ov = int(round(route['Forenzen_OV'] * SIMULATIE_SCHAAL_FACTOR))
    
    if totaal_ov <= 0:
        continue
        
    origin_stations = station_weights[station_weights['gemeente_naam'] == origin]
    dest_stations = station_weights[station_weights['gemeente_naam'] == dest]
    
    if origin_stations.empty or dest_stations.empty:
        continue
        
    for i in range(totaal_ov):
        orig_row = origin_stations.sample(n=1, weights='Station_Weight').iloc[0]
        dest_row = dest_stations.sample(n=1, weights='Station_Weight').iloc[0]
        
        # Heenreis
        rand_heen = random.random() 
        if rand_heen <= P_HEEN_OCHTEND:
            periode_heen = "Ochtendspits"
            spawn_heen = int(np.random.normal(loc=27900, scale=1800))
            spawn_heen = max(25200, min(32400, spawn_heen))
        elif rand_heen <= (P_HEEN_OCHTEND + P_HEEN_DAL):
            periode_heen = "Daluren"
            spawn_heen = random.randint(32400, 57600)
        else:
            periode_heen = "Avondspits"
            spawn_heen = random.randint(57600, 64800)
            
        passagiers_lijst.append({
            'Passagier_ID': len(passagiers_lijst) + 1,
            'Gemeente_Oorsprong': origin,
            'Gemeente_Bestemming': dest,
            'Start_Station': orig_row['Nearest_Station_Name'],
            'Eind_Station': dest_row['Nearest_Station_Name'],
            'Tijdsperiode': periode_heen,
            'Spawn_Tijd_Sec': spawn_heen,
            'Reis_Type': 'Heenreis'
        })
        
        # Terugreis
        rand_terug = random.random()
        if rand_terug <= P_TERUG_OCHTEND:
            periode_terug = "Ochtendspits"
            spawn_terug = random.randint(25200, 32400)
        elif rand_terug <= (P_TERUG_OCHTEND + P_TERUG_DAL):
            periode_terug = "Daluren"
            spawn_terug = random.randint(32400, 57600)
        else:
            periode_terug = "Avondspits"
            spawn_terug = int(np.random.normal(loc=61200, scale=1800)) # Piek rond 17:00
            spawn_terug = max(57600, min(66600, spawn_terug))
            
        # Voorkom dat mensen terugreizen voordat ze arriveren
        spawn_terug = max(spawn_heen + 14400, spawn_terug)
        if spawn_terug > 86399:
            spawn_terug = 86399
            
        passagiers_lijst.append({
            'Passagier_ID': len(passagiers_lijst) + 1,
            'Gemeente_Oorsprong': dest, # Woon/Werk omgedraaid
            'Gemeente_Bestemming': origin,
            'Start_Station': dest_row['Nearest_Station_Name'], # Route omgedraaid
            'Eind_Station': orig_row['Nearest_Station_Name'],
            'Tijdsperiode': periode_terug,
            'Spawn_Tijd_Sec': spawn_terug,
            'Reis_Type': 'Terugreis'
        })

df_passagiers = pd.DataFrame(passagiers_lijst)
if not df_passagiers.empty:
    # Friese Halte Check
    puur_friese_haltenamen = friese_ov_stations_puur['stop_name'].unique()
    df_passagiers = df_passagiers[
        df_passagiers['Start_Station'].isin(puur_friese_haltenamen) | 
        df_passagiers['Eind_Station'].isin(puur_friese_haltenamen)
    ]

    df_passagiers = df_passagiers.sort_values(by='Spawn_Tijd_Sec').reset_index(drop=True)
    print(f"Er zijn {len(df_passagiers)} OV-passagiers gegenereerd voor de gehele dag.")
    print("\nVerdeling per tijdsperiode (Heen + Terug):")
    print(df_passagiers['Tijdsperiode'].value_counts())
    print("\nVoorbeeld van routeplannen van reizigers:")
    print(df_passagiers.head(10).to_string(index=False))
else:
    print("Geen passagiers gegenereerd.")

In deze cel wordt het netwerk opgebouwd om in SimPy de simulatie te kunnen runnen. De GTFS data wordt nogmaals ingeladen en gefilterd en de simulatie wordt opgebouwd (inclusief: Dijkstra algoritme (routeplanner), stations/haltes, treinen & bussen, virtuele passagiers).

In [ ]:
# GTFS Dienstregeling voorbereiden voor SimPy

# GTFS data inladen
routes = pd.read_csv(path_to_gtfs + '/routes.txt')
trips = pd.read_csv(path_to_gtfs + '/trips.txt')
calendar_dates = pd.read_csv(path_to_gtfs + '/calendar_dates.txt')
stop_times = pd.read_csv(path_to_gtfs + '/stop_times.txt', dtype={'stop_id': str, 'trip_id': str})

routes['route_id'] = routes['route_id'].astype(str)
trips['route_id'] = trips['route_id'].astype(str)
trips['service_id'] = trips['service_id'].astype(str)
trips['trip_id'] = trips['trip_id'].astype(str)
calendar_dates['service_id'] = calendar_dates['service_id'].astype(str)

# Relevante routes uitzoeken
ov_routes = routes[routes['route_type'].isin([2, 3, 700, 701, 702, 703, 704, '2', '3', '700'])]
active_services = calendar_dates[(calendar_dates['date'] == int(SIM_DATE)) & (calendar_dates['exception_type'] == 1)]['service_id'].tolist()
active_services = [str(x) for x in active_services]

active_trips = trips[(trips['service_id'].isin(active_services)) & (trips['route_id'].isin(ov_routes['route_id']))]
active_trips = active_trips.merge(ov_routes[['route_id', 'agency_id', 'route_type']], on='route_id', how='left')

# Capaciteit bepalen
def bepalen_capaciteit(row):
    r_type = str(row['route_type'])
    agency = str(row['agency_id']).upper()
    if r_type in ['3', '700']:
        return 60 
    if 'ARRIVA' in agency:
        return 150 
    elif 'NS' in agency:
        return 400 
    else:
        return 60

active_trips['Capaciteit'] = active_trips.apply(bepalen_capaciteit, axis=1)

friese_stops = friese_ov_stations['stop_id'].astype(str).tolist()

active_st = stop_times[stop_times['trip_id'].isin(active_trips['trip_id'])]
st_in_friesland = active_st[active_st['stop_id'].isin(friese_stops)]
trips_in_friesland = st_in_friesland['trip_id'].value_counts()
valid_trips = trips_in_friesland[trips_in_friesland > 1].index.tolist()

final_st = active_st[(active_st['trip_id'].isin(valid_trips)) & (active_st['stop_id'].isin(friese_stops))]

def time_to_seconds(t_str):
    if pd.isna(t_str): return 0
    h, m, s = map(int, str(t_str).split(':'))
    return h * 3600 + m * 60 + s

final_st = final_st.copy()
final_st['arrival_sec'] = final_st['arrival_time'].apply(time_to_seconds)
final_st['departure_sec'] = final_st['departure_time'].apply(time_to_seconds)

final_st = final_st.sort_values(by=['trip_id', 'stop_sequence'])
final_st = final_st.merge(friese_ov_stations[['stop_id', 'stop_name']], on='stop_id', how='left')
active_trips_dict = active_trips.set_index('trip_id')['Capaciteit'].to_dict()

print(f"Gevonden OV-ritten: {len(valid_trips)}")
print(f"\nTotaal aantal OV-stops voor de simulatie: {len(final_st)}\n")
print("\nStap 3.2: SimPy Engine & Tijdsgebonden Dijkstra Reisplanner Opzetten...")

# Fries OV-Netwerk bouwen
station_departures = {}
for trip_id, stops_df in final_st.groupby('trip_id'):
    st_list = stops_df.sort_values('stop_sequence').to_dict('records')
    for idx in range(len(st_list) - 1):
        u = st_list[idx]['stop_name']
        v = st_list[idx+1]['stop_name']
        dep_u = st_list[idx]['departure_sec']
        arr_v = st_list[idx+1]['arrival_sec']
        if u not in station_departures:
            station_departures[u] = []
        station_departures[u].append((dep_u, arr_v, v, trip_id))

# Sorteer departures per station op vertrektijd voor snellere zoektijd
for u in station_departures:
    station_departures[u].sort(key=lambda x: x[0])

print(f"Netwerk gebouwd met {len(station_departures)} OV-haltes.")

# Simulatie definieren
class DUS_Simulatie:
    def __init__(self, env, stations_list):
        self.env = env
        self.stations = {s: [] for s in stations_list}
        self.log_data = [] 
        
    def log(self, tijd, event_type, station, passagiers_aantal, trein_id=None, trein_bezetting=None, capaciteit=None):
        self.log_data.append({
            'Tijd_Sec': tijd,
            'Event': event_type,
            'Station': station,
            'Aantal_Passagiers': passagiers_aantal,
            'Trein_ID': trein_id,
            'Trein_Bezetting': trein_bezetting,
            'Trein_Capaciteit': capaciteit
        })

# Dijkstra algoritme implementeren (routeplanner)
def time_dependent_dijkstra(start_station, eind_station, start_time, departures_dict):
    pq = [(start_time, start_station)] 
    earliest_arrival = {start_station: start_time}
    parent = {}
    
    while pq:
        time, u = heapq.heappop(pq)
        
        if time > earliest_arrival.get(u, float('inf')):
            continue
            
        if u == eind_station:
            break
            
        for dep_time, arr_time, v, trip_id in departures_dict.get(u, []):
            if dep_time >= time: 
                if arr_time < earliest_arrival.get(v, float('inf')):
                    earliest_arrival[v] = arr_time
                    parent[v] = u
                    heapq.heappush(pq, (arr_time, v))
                    
    if eind_station not in earliest_arrival:
        return []
        
    path = [eind_station]
    curr = eind_station
    while curr != start_station:
        curr = parent[curr]
        path.append(curr)
    path.reverse()
    
    # Filter opeenvolgende duplicates
    clean_path = [path[0]]
    for p in path[1:]:
        if p != clean_path[-1]:
            clean_path.append(p)
    return clean_path

# Passagiers genereren
def passagier_generator(env, sim, passagiers_df, departures_dict):
    for _, reiziger in passagiers_df.iterrows():
        wachttijd = reiziger['Spawn_Tijd_Sec'] - env.now
        if wachttijd > 0:
            yield env.timeout(wachttijd)
            
        start_station = reiziger['Start_Station']
        eind_station = reiziger['Eind_Station']
        
        if start_station == eind_station:
            continue
            
        route_plan = time_dependent_dijkstra(start_station, eind_station, env.now, departures_dict)
        
        if not route_plan:
            sim.log(env.now, 'Geen Route Mogelijk', start_station, 1)
            continue 
            
        if start_station in sim.stations:
            sim.stations[start_station].append({
                'id': reiziger['Passagier_ID'],
                'route_plan': route_plan,
                'bestemming': eind_station
            })
            sim.log(env.now, 'Aankomst Perron', start_station, len(sim.stations[start_station]))

# Virtuele treinen genereren
def trein_proces(env, sim, trip_id, capaciteit, haltes_df):
    passagiers_in_trein = []
    haltes = haltes_df[haltes_df['trip_id'] == trip_id].sort_values(by='stop_sequence')
    
    for i, halte in haltes.iterrows():
        station_naam = halte['stop_name']
        aankomst = halte['arrival_sec']
        vertrek = halte['departure_sec']
        
        rijtijd = aankomst - env.now
        if rijtijd > 0:
            yield env.timeout(rijtijd)
            
        uitstappers = [p for p in passagiers_in_trein if p.get('current_doel') == station_naam]
        passagiers_in_trein = [p for p in passagiers_in_trein if p.get('current_doel') != station_naam]
        
        if len(uitstappers) > 0:
            sim.log(env.now, 'Uitstappen', station_naam, len(uitstappers), trip_id, len(passagiers_in_trein), capaciteit)
            
            bestemming_bereikt = 0
            for p in uitstappers:
                if p['bestemming'] != station_naam:
                    if station_naam in sim.stations:
                        sim.stations[station_naam].append(p)
                        sim.log(env.now, 'Overstap Wachtrij', station_naam, 1)
                else:
                    bestemming_bereikt += 1
            
            if bestemming_bereikt > 0:
                sim.log(env.now, 'Aankomst Bestemming', station_naam, bestemming_bereikt)
                
        wachttijd = vertrek - env.now
        if wachttijd > 0:
            yield env.timeout(wachttijd) 
            
        if station_naam in sim.stations:
            wachtrij = sim.stations[station_naam]
            rest_haltes = haltes[haltes['stop_sequence'] > halte['stop_sequence']]['stop_name'].tolist()
            
            wachtrij_over = []
            instappers = 0
            
            for p in wachtrij:
                route_plan = p['route_plan']
                try:
                    huidige_index = route_plan.index(station_naam)
                    volgende_station = route_plan[huidige_index + 1]
                except (ValueError, IndexError):
                    wachtrij_over.append(p)
                    continue
                
                if volgende_station in rest_haltes:
                    if len(passagiers_in_trein) < capaciteit:
                        rest_route = route_plan[huidige_index + 1:]
                        current_doel = None
                        for station_in_route in reversed(rest_route):
                            if station_in_route in rest_haltes:
                                current_doel = station_in_route
                                break
                        
                        p['current_doel'] = current_doel
                        passagiers_in_trein.append(p)
                        instappers += 1
                    else:
                        wachtrij_over.append(p) 
                else:
                    wachtrij_over.append(p) 
                    
            sim.stations[station_naam] = wachtrij_over
            
            if instappers > 0:
                sim.log(env.now, 'Instappen', station_naam, instappers, trip_id, len(passagiers_in_trein), capaciteit)

print("SimPy OV-Netwerk gebouwd is gebouwd.")

In deze cel wordt het hele opgebouwde SimPy simulatiemodel uitgevoerd. Daarnaast worden logs verwerkt en worden meteen knelpunten (vraag hoger dan capaciteit) van het OV-netwerk uitgeprint.

In [ ]:
# SimPy Model Uitvoeren

env = simpy.Environment(initial_time=START_TIME)

alle_stations = friese_ov_stations['stop_name'].unique().tolist()
sim = DUS_Simulatie(env, alle_stations)

# Passagier-generatie toevoegen
env.process(passagier_generator(env, sim, df_passagiers, station_departures))

# Treinen toevoegen als processen
for trip_id in valid_trips:
    capaciteit = active_trips_dict.get(trip_id, 300)
    env.process(trein_proces(env, sim, trip_id, capaciteit, final_st))

print(f"Simulatie gestart! Totaal aantal ritten: {len(valid_trips)}")
print(f"Passagiers verwacht op de perrons: {len(df_passagiers)}")

# Run Simulatie
env.run()

print("\nSimulatie Voltooid!")
print(f"Eindtijd simulatie: {int(env.now // 3600)}:{int((env.now % 3600) // 60):02d}")

# Verwerken van logs
log_df = pd.DataFrame(sim.log_data)
if not log_df.empty:    
    totaal_aangekomen = log_df[log_df['Event'] == 'Aankomst Perron']['Aantal_Passagiers'].sum()
    totaal_ingestapt = log_df[log_df['Event'] == 'Instappen']['Aantal_Passagiers'].sum()
    totaal_overstappers = log_df[log_df['Event'] == 'Overstap Wachtrij']['Aantal_Passagiers'].sum()
    totaal_bestemming = log_df[log_df['Event'] == 'Aankomst Bestemming']['Aantal_Passagiers'].sum()
    
    print(f"Totaal instap-momenten (inclusief overstaps): {totaal_ingestapt}")
    print(f"Totaal virtuele overstappen geregistreerd: {totaal_overstappers}")
    print(f"Totaal unieke reizigers succesvol op eindbestemming: {totaal_bestemming} / {len(df_passagiers)}")
    
    # Knelpunten zoeken
    volle_treinen = log_df[log_df['Trein_Bezetting'] >= log_df['Trein_Capaciteit']]
    if not volle_treinen.empty:
        print(f"\nEr waren {len(volle_treinen)} momenten waarop een trein zijn maximale capaciteit bereikte.")
        print(volle_treinen.sort_values(by='Trein_Bezetting', ascending=False).head())
    else:
        print("\nGeen capaciteitsknelpunten gevonden. Alle reizigers konden hun treinen halen.")
else:
    print("Geen logs gegenereerd.")

### 5. Auto Simulatie

In deze cel wordt het autonetwerk gemaakt en bepaald hoevaak een weg (edge) wordt bereden. Het simuleert autoverkeer door op basis van de modal split en CBS data virtuele auto's te laten rijden van origin naar destination.

In [ ]:
# Auto Simulatie

buurten_wgs84 = buurten_fryslan_gdf.to_crs(epsg=4326).copy()

# Buurten organiseren
gemeente_buurten = {}
for row in buurten_wgs84.itertuples():
    gem, code, pop = row.gemeente_naam.strip(), row.buurt_code, row.aantal_inwoners
    if pop > 0:
        if gem not in gemeente_buurten: gemeente_buurten[gem] = []
        gemeente_buurten[gem].append({'code': code, 'pop': pop, 'point': row.geometry.centroid})

# Koppeling woonregio's met autonetwerk
buurt_nodes_car = {}
relevante_gemeenten = set(simulatie_vraag['WoonregioS'].unique()).union(set(simulatie_vraag['WerkregioS'].unique()))
for gem, buurten in gemeente_buurten.items():
    if gem in relevante_gemeenten:
        for b in buurten: buurt_nodes_car[b['code']] = ox.distance.nearest_nodes(G_drive, X=b['point'].x, Y=b['point'].y)

edge_counts = Counter()
auto_routes_df = simulatie_vraag[simulatie_vraag['Forenzen_Auto'] > 0]

# Routes bepalen
for row in tqdm(list(auto_routes_df.itertuples()), desc='Simuleren autoverkeer'):
    woon, werk, aantal_autos = row.WoonregioS.strip(), row.WerkregioS.strip(), int(round(row.Forenzen_Auto * SIMULATIE_SCHAAL_FACTOR))
    if aantal_autos > 0 and woon in gemeente_buurten and werk in gemeente_buurten:
        aantal_paden = min(aantal_autos, 5) 
        autos_per_pad, rest_autos = aantal_autos // aantal_paden, aantal_autos % aantal_paden
        woon_buurten, werk_buurten = gemeente_buurten[woon], gemeente_buurten[werk]
        woon_weights, werk_weights = [b['pop'] for b in woon_buurten], [b['pop'] for b in werk_buurten]
        
        if sum(woon_weights) == 0 or sum(werk_weights) == 0: continue
            
        for p in range(aantal_paden):
            source_b = random.choices(woon_buurten, weights=woon_weights, k=1)[0]
            target_b = random.choices(werk_buurten, weights=werk_weights, k=1)[0]
            source_node, target_node = buurt_nodes_car.get(source_b['code']), buurt_nodes_car.get(target_b['code'])
            flow = autos_per_pad + (rest_autos if p == 0 else 0)
            
            if source_node and target_node and source_node != target_node:
                try:
                    path = nx.shortest_path(G_drive, source=source_node, target=target_node, weight='length')
                    # Sneller tellen via zip generator en Python Counter
                    edges_in_path = list(zip(path[:-1], path[1:]))
                    for edge in edges_in_path:
                        edge_counts[edge] += flow
                except nx.NetworkXNoPath:
                    pass

# Resultaten exporteren
export_folder = 'Simulatie_Resultaten'
os.makedirs(export_folder, exist_ok=True)

auto_export_data = []
for (u, v), flow in edge_counts.items():
    edge_data = G_drive.get_edge_data(u, v)[0]
    name = edge_data.get('name', 'Onbekende Weg')
    name = name[0] if isinstance(name, list) else name
    speed = edge_data.get('maxspeed', 50)
    speed = speed[0] if isinstance(speed, list) else speed
    auto_export_data.append({'Van_Node': u, 'Naar_Node': v, 'Straatnaam': name, 'Lengte_m': round(edge_data.get('length', 0), 1), 'Snelheid_kmu': speed, 'Volume_Autos': int(round(flow / SIMULATIE_SCHAAL_FACTOR))})
        
pd.DataFrame(auto_export_data).to_csv(os.path.join(export_folder, 'auto_edge_counts.csv'), index=False)
log_df.to_csv(os.path.join(export_folder, 'OV_Ruwe_Logs.csv'), index=False)
simulatie_vraag.to_csv(os.path.join(export_folder, 'Totale_Vraag_Modal_Split.csv'), index=False)

if not log_df.empty:
    station_activity = log_df[log_df['Event'].isin(['Instappen', 'Uitstappen'])].groupby('Station')['Aantal_Passagiers'].sum().reset_index()
    station_activity['Aantal_Passagiers'] = (station_activity['Aantal_Passagiers'] / SIMULATIE_SCHAAL_FACTOR).round().astype(int)
    station_activity.to_csv(os.path.join(export_folder, 'ov_station_drukte.csv'), index=False)

### 6. Validatie (Autonetwerk)

In deze cel worden de uitslagen van de simulatie gevalideert tegenover data van het Nationaal Dataportaal Wegverkeer (NDW). Deze cel vormt een correlatietest die het gesimuleerde volume autoverkeer van een weg (edge) koppelt met een meetpaal uit de NDW data. Dit volume wordt vergeleken en op basis daarvan wordt de correlatiescore bepaald. Het probeert rekening te houden met het feit dat 'avondspits_csv' slechts metingen zijn van één uur tijdens de avondspits via de parameter 'AANDEEL_SPITSUUR_AUTO' (10%).

In [ ]:
# Validatie (Auto-)Simulatie met avondspits.csv

avondspits_df = pd.read_csv(path_to_val)
y_true = []
y_pred = []
    
for row in avondspits_df.itertuples():
    # Zoek dichtstbijzijnde edge
    u, v, key = ox.distance.nearest_edges(G_drive, X=row.longitude, Y=row.latitude)
    # Tel volume in beide richtingen op
    sim_flow = edge_counts.get((u, v), 0) + edge_counts.get((v, u), 0)
    sim_flow_spits = sim_flow * AANDEEL_SPITSUUR_AUTO
        
    y_true.append(row.verkeersintensiteit)
    y_pred.append(sim_flow_spits)
                
# Plot
plt.figure(figsize=(10,6))
plt.scatter(y_true, y_pred, alpha=0.5, color='purple', edgecolors='white', s=50)
        
# Lijn voor perfecte correlatie
max_val = max(max(y_true), max(y_pred))
plt.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='Perfecte Match')
        
plt.title('Validatie: avondspits.csv vs Simulatie (Auto)')
plt.xlabel('Echte Verkeersintensiteit (Meetpaal)')
plt.ylabel('Gesimuleerd Volume (Spits)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
        
# Correlatie Berekenen
cor = np.corrcoef(y_true, y_pred)[0,1]
print(f"Correlatiecoëfficiënt (Pearson r): {cor:.2f}")

Uit de correlatie test en bijbehorende score (0.17) is gebleken dat de simulatie niet foutloos is en de verschillen tussen ‘avondspits.csv’ en de resultaten van het model te groot zijn. Echter zijn er goede redenen voor deze lage score:
- De simulatie berekent slechts woon-werkverkeer op basis van de CBS dataset. De meetpalen in ‘avondspits.csv’ meten al het verkeer op de weg.
- Het Dijkstra algoritme is niet geavanceerd genoeg; Het kan niet rekening houden met het scenario dat een automobilist niet altijd de wiskundig kortste route neemt.
- Sommige meetpalen meten slechts het verkeer een kant op, terwijl de simulatie cijfers altijd twee kanten op zijn
- De parameters van de modal split en AANDEEL_SPITSUUR_AUTO zijn aanpasbaar en niet perfect

### 7. Interactieve Kaart

In deze cel wordt de interactieve kaart opgebouwd met behulp van de Folium library. De kaart bestaat uit drie lagen:
- Station Bubbels, haltes/stations gerepresenteerd door color-coded bubbels met het aantal in/uitstappers (/overstappers!) op de gesimuleerde dag
- AntPaths, visualiseren de top 100 theoretische reisstromen tussen gemeenten door middel van neon lijnen die toenemen in dikte naarmate volume voorspelde OV-reizigers toenemen.
- Autonetwerk, de getekende routes op de kaart met color-coded wegen (edges) op basis van het volume auto's

In [ ]:
# Folium Interactieve Kaart

m = folium.Map(location=[53.164164, 5.781754], zoom_start=10, tiles='cartodbdark_matter')

# OV Stations (Bubbels)
fg_stations = folium.FeatureGroup(name="Station Drukte (Bubbels)")
if not log_df.empty:
    station_coords = friese_ov_stations[['stop_name', 'stop_lat', 'stop_lon']].drop_duplicates(subset=['stop_name'])
    st_act = log_df[log_df['Event'].isin(['Instappen', 'Uitstappen'])].groupby('Station')['Aantal_Passagiers'].sum().reset_index()
    st_act = st_act.merge(station_coords, left_on='Station', right_on='stop_name', how='inner')
    
    if not st_act.empty:
        p98, p90, p75 = np.percentile(st_act['Aantal_Passagiers'], [98, 90, 75])
        for row in st_act.itertuples():
            pax = int(round(row.Aantal_Passagiers / SIMULATIE_SCHAAL_FACTOR))
            radius = min(35, max(4, np.log1p(pax) * 3.5)) # Logaritmische schaal
            if pax >= p98: color = '#ff0000'
            elif pax >= p90: color = '#ff9900'
            elif pax >= p75: color = '#ffff00'
            else: color = '#00ff00'
                
            folium.CircleMarker(
                location=[float(row.stop_lat), float(row.stop_lon)], radius=radius,
                popup=f"{row.Station}: {pax} in/uitstappers",
                color=color, fill=True, fill_color=color, fill_opacity=0.6, weight=1
            ).add_to(fg_stations)
m.add_child(fg_stations)

# OV Pendelstromen (Top 100 AntPaths)
fg_flows = folium.FeatureGroup(name="OV Pendelstromen (Top 100)")
top_flows = simulatie_vraag.sort_values('Forenzen_OV', ascending=False).head(100)

DUS_regios = municipalities[municipalities['statnaam'].isin(alle_toegestane_gemeenten)].copy()
DUS_copy = DUS_regios.to_crs(epsg=28992)
DUS_copy['centroid'] = DUS_copy.geometry.centroid
DUS_centroids_wgs84 = DUS_copy['centroid'].to_crs(epsg=4326)
gemeente_coords = {statnaam.strip(): (point.y, point.x) for statnaam, point in zip(DUS_copy['statnaam'], DUS_centroids_wgs84)}

for row in top_flows.itertuples():
    woon, werk, passagiers = row.WoonregioS.strip(), row.WerkregioS.strip(), row.Forenzen_OV
    if passagiers > 0 and woon in gemeente_coords and werk in gemeente_coords:
        weight = min(8, max(2, passagiers / 50))
        plugins.AntPath(
            locations=[gemeente_coords[woon], gemeente_coords[werk]],
            popup=f"{woon} -> {werk}: {passagiers} reizigers",
            color="#00e5ff", pulse_color="#ffffff", weight=weight, delay=800, dash_array=[15, 30]
        ).add_to(fg_flows)
m.add_child(fg_flows)

# Auto Simulatie (Wegennetwerk)
fg_auto = folium.FeatureGroup(name="Gesimuleerd Autoverkeer (Top Wegen)")
if len(edge_counts) > 0:
    max_flow = max(edge_counts.values())
    norm = mcolors.LogNorm(vmin=1, vmax=max_flow)
    cmap = plt.get_cmap('plasma') # Deprecation warning gefixt
        
    for (u, v), flow in edge_counts.items():
        if flow > 0:
            lat_u, lon_u = G_drive.nodes[u]['y'], G_drive.nodes[u]['x']
            lat_v, lon_v = G_drive.nodes[v]['y'], G_drive.nodes[v]['x']
            color = mcolors.to_hex(cmap(norm(flow)))
            weight = min(max((flow / max_flow) * 15, 3), 12)
            
            folium.PolyLine(
                locations=[(lat_u, lon_u), (lat_v, lon_v)],
                color=color, weight=weight, opacity=0.8,
                tooltip=f"Verkeersvolume: {int(round(flow / SIMULATIE_SCHAAL_FACTOR))} auto's",
                popup=folium.Popup(f"<b>Gesimuleerd Volume:</b><br>{int(round(flow / SIMULATIE_SCHAAL_FACTOR))} auto's op dit wegsegment", max_width=250)
            ).add_to(fg_auto)
m.add_child(fg_auto)

folium.LayerControl().add_to(m)
map_path = os.path.join(export_folder, "Friesland_Simulatie_Map.html")
m.save(map_path)
print(f"Interactieve map opgeslagen als: {map_path}")

### 8. Resultaten

Deze cel exporteert OV trajecten en print de drukste (auto-)kenlpunten en traagste verbindingen.

In [ ]:
# Analyse (Knelpunten & Traagste Routes)

# Printen van drukste knelpunten
auto_df = pd.DataFrame(auto_export_data) 
if not auto_df.empty:
    print("Top 10 Drukste Auto Knelpunten (Volume Auto's)")
    top_knelpunten = auto_df.sort_values('Volume_Autos', ascending=False).head(10)
    print(top_knelpunten[['Straatnaam', 'Lengte_m', 'Snelheid_kmu', 'Volume_Autos']].to_string(index=False))

# Traagste pendelverbindingen tussen gemeenten
print("\nTop 10 Traagste Pendelverbindingen (Hoogste Reistijd per km)")
analyse_df = simulatie_vraag.copy()
analyse_df['Auto_Min_per_km'] = analyse_df['Tijd_Auto'] / analyse_df['Afstand_km']
analyse_df['OV_Min_per_km'] = analyse_df['Tijd_OV'] / analyse_df['Afstand_km']

trage_auto = analyse_df[analyse_df['Forenzen_Auto'] > 10].sort_values('Auto_Min_per_km', ascending=False)
print("   Auto:")
print(trage_auto[['WoonregioS', 'WerkregioS', 'Afstand_km', 'Tijd_Auto', 'Auto_Min_per_km']].head(10).to_string(index=False))

trage_ov = analyse_df[analyse_df['Forenzen_OV'] > 10].sort_values('OV_Min_per_km', ascending=False)
print("   OV:")
print(trage_ov[['WoonregioS', 'WerkregioS', 'Afstand_km', 'Tijd_OV', 'OV_Min_per_km']].head(10).to_string(index=False))

# Extra export (OV trajecten)
if not log_df.empty:
    ov_trajecten = log_df[log_df['Event'].isin(['Instappen', 'Uitstappen'])].groupby(['Trein_ID', 'Station', 'Event'])['Aantal_Passagiers'].sum().reset_index()
    ov_trajecten['Aantal_Passagiers'] = (ov_trajecten['Aantal_Passagiers'] / SIMULATIE_SCHAAL_FACTOR).round().astype(int)
    ov_trajecten.to_csv(os.path.join(export_folder, 'OV_Traject_Tellingen.csv'), index=False)
    print(f"\nOV traject tellingen geëxporteerd naar: {export_folder}/OV_Traject_Tellingen.csv")

print("DUS Simulatie Friesland Voltooid!")

### 9. Sterke en Zwakke Punten

De Simulatie heeft een aantal erg duidelijke sterke en zwakke punten. Een aantal sterke punten zijn:
- De simulatie is flexibel, het kan verschillende modal splits simuleren op basis van de parameters.
- De simulatie is tot op zekere hoogte transparant. De formules en models zitten logisch in elkaar en passagieraantallen kunnen nagerekend worden.
- De simulatie weerspiegelt de werkelijkheid zoals hij is door gebruik te maken van betrouwbare databronnen, dienstregelingen en wegenkaarten.
- De simulatie kan goed worden gebruikt om knelpunten te visualiseren en berekenen.
- De simulatie werkt op basis van openbare data. Iedereen kan bij deze data en het model runnen.
- De simulatie is in theorie toepasbaar op andere regio's. Dit hangt wel af van de beschikbare data voor die regio.

Daarnaast heeft de simulatie ook een paar duidelijke zwakke punten:
- De simulatie runt op basis van schattingen van parameters (bijv. tijdsprofiel, nutfuncties en kosten).
- De simulatie werkt op basis van CBS woon-werkverkeer en neemt dus geen studenten, kinderen, ouderen, vakantiegangers, recreatief verkeer, enzovoort mee.
- De simulatie is statisch in de zin dat deze momenteel geen mogelijkheid biedt nieuwe wegen/spoorlijnen/busroutes toe te voegen.
- De simulatie werkt op basis van een CBS dataset op gemeenteniveau. Woon- en werkregio zijn op gemeenteniveau, waaraan wijken en buurten gekoppeld worden om vertrek/aankomst stations/haltes/autowegen te voorspellen.
- De validatie laat zien dat het model niet erg hoog correleert met de werkelijkheid, vooral het niveau waarop (openbare) databronnen beschikbaar waren zijn hiervan de oorzaak. Ook zou een dataset van een complete (werk)dag een betere maatstaaf zijn.